# 🧪 Testing Pipeline de Deals Detection

Este notebook ejecuta el flujo completo:
1. Extracción de datos (query_historicos.py)
2. Pipeline de baselines (pipeline_build_baselines.py)
3. Validación de resultados
4. Test de clasificación

In [1]:
import pandas as pd
import numpy as np
import sys
import os
from pathlib import Path

# Agregar directorio raíz al path
PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

# Importar módulos propios
import config
import auxiliary_functions

# Configurar pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print("✅ Imports completados")
print(f"📁 Directorio base: {config.BASE_DIR}")

✅ Imports completados
📁 Directorio base: /home/martin/Desktop/hotels/deals_analysis


## 1️⃣ Verificar Rutas y Archivos

In [2]:
# Verificar directorios
print("=" * 60)
print("VERIFICACIÓN DE DIRECTORIOS")
print("=" * 60)

directories = [
    ("Data", config.DATA_DIR),
    ("Price Historicals", config.PRICE_HISTORICALS_DIR),
    ("Outputs", config.OUTPUT_DIR),
]

for name, path in directories:
    exists = "✅" if path.exists() else "❌"
    print(f"{exists} {name}: {path}")

# Verificar archivos clave
print("\n" + "=" * 60)
print("ARCHIVOS CLAVE")
print("=" * 60)

files = [
    ("Destination Mapping", config.DESTINATION_MAPPING_FILE),
    ("Baselines Output", config.BASELINES_FILE),
]

for name, path in files:
    exists = "✅" if path.exists() else "⚠️ (se creará)"
    print(f"{exists} {name}: {path}")

# Listar archivos históricos existentes
print("\n" + "=" * 60)
print("ARCHIVOS HISTÓRICOS EXISTENTES")
print("=" * 60)

if config.PRICE_HISTORICALS_DIR.exists():
    csv_files = list(config.PRICE_HISTORICALS_DIR.glob('*.csv'))
    if csv_files:
        for f in sorted(csv_files):
            size_mb = f.stat().st_size / (1024 * 1024)
            print(f"  📄 {f.name} ({size_mb:.2f} MB)")
    else:
        print("  ⚠️ No hay archivos CSV")
else:
    print("  ❌ Directorio no existe")

VERIFICACIÓN DE DIRECTORIOS
✅ Data: /home/martin/Desktop/hotels/deals_analysis/data
✅ Price Historicals: /home/martin/Desktop/hotels/deals_analysis/data/price_historicals
✅ Outputs: /home/martin/Desktop/hotels/deals_analysis/outputs

ARCHIVOS CLAVE
✅ Destination Mapping: /home/martin/Desktop/hotels/deals_analysis/data/destination_with_nearest.csv
⚠️ (se creará) Baselines Output: /home/martin/Desktop/hotels/deals_analysis/outputs/market_baselines.csv

ARCHIVOS HISTÓRICOS EXISTENTES
  ⚠️ No hay archivos CSV


## 2️⃣ Extracción de Datos (Query 2025)

**Nota:** Esta celda ejecutará la query para extraer datos de 2025. Comentar si ya tienes los datos.

In [3]:
# Ejecutar query para 2025
import query_historicos

print("🔄 Ejecutando query para 2025...")
try:
    query_historicos.main(2025)
    print("\n✅ Query completada exitosamente")
except Exception as e:
    print(f"\n❌ Error en query: {e}")
    print("\n⚠️ Si no tienes acceso a DB, copia manualmente historicals_2025.csv a data/price_historicals/")

🔄 Ejecutando query para 2025...
EXTRACCIÓN DE DATOS HISTÓRICOS - AÑO 2025
✓ Conexión exitosa a la base de datos

Ejecutando query para año 2025...


/home/martin/Desktop/hotels/deals_analysis/query_historicos.py:83: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, cnx)


❌ Error al ejecutar query: Execution failed on sql: 
    SELECT 
        h.city, 
        h.state, 
        h.country, 
        h.country_code, 
        a.date_start, 
        a.date_end, 
        a.number_of_adults, 
        a.number_of_rooms,
        a.number_of_kids, 
        a.nights, 
        COUNT(*) AS count_repeated,  
        AVG(a.hotel_count) AS avg_hotel_count,
        MIN(a.hotel_count) AS min_hotel_count,
        MAX(a.hotel_count) AS max_hotel_count,
        AVG(a.price_average) AS avg_price_average,
        MAX(a.price_high) AS max_price_high,
        MIN(a.price_low) AS min_price_low
    FROM 
        analytic.customer_shopping_model AS a
    JOIN 
        analytic.hotel_city_location AS h
    ON 
        a.hotel_id = h.hotel_id
    WHERE 
        a.date_start >= '2025-01-01' 
        AND a.date_start < '2026-01-01'
        AND a.date_end >= '2025-01-01' 
        AND a.date_end < '2026-01-01'
        AND a.type = 'HOTELS'
    GROUP BY 
        h.city, 
        a.date_s

SystemExit: 1

/home/martin/miniconda3/envs/revmax/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


### ⚠️ Crear Datos de Muestra (Si la query falla)

Si la query falló por timeout, ejecuta esta celda para crear datos de muestra:

## 3️⃣ Validar Datos Extraídos

In [6]:
# Cargar y explorar datos históricos
print("=" * 60)
print("EXPLORACIÓN DE DATOS HISTÓRICOS")
print("=" * 60)

try:
    df_raw = auxiliary_functions.load_all_historicals()
    
    print(f"\n📊 Dimensiones: {df_raw.shape}")
    print(f"\n📅 Rango de fechas:")
    print(f"  Inicio: {df_raw['date_start'].min()} → {df_raw['date_start'].max()}")
    print(f"  Fin:    {df_raw['date_end'].min()} → {df_raw['date_end'].max()}")
    
    print(f"\n🌍 Geografía:")
    print(f"  Países: {df_raw['country'].nunique()}")
    print(f"  Estados: {df_raw['state'].nunique()}")
    print(f"  Ciudades: {df_raw['city'].nunique()}")
    
    print(f"\n🔢 Estadísticas básicas:")
    print(f"  Búsquedas repetidas (sum): {df_raw['count_repeated'].sum():,}")
    print(f"  Precio promedio (mean): ${df_raw['avg_price_average'].mean():.2f}")
    print(f"  Noches (median): {df_raw['nights'].median():.0f}")
    
    print(f"\n📋 Primeras 5 filas:")
    display(df_raw.head())
    
    print(f"\n🔍 Columnas disponibles:")
    print(df_raw.columns.tolist())
    
except Exception as e:
    print(f"❌ Error cargando datos: {e}")
    df_raw = None

EXPLORACIÓN DE DATOS HISTÓRICOS

Cargando 1 archivos históricos...
  ✓ historicals_2025.csv: 5,000 registros

✓ Total cargado: 5,000 registros

📊 Dimensiones: (5000, 18)

📅 Rango de fechas:
  Inicio: 2025-01-01 → 2025-12-23
  Fin:    2025-01-02 → 2025-12-30

🌍 Geografía:
  Países: 1
  Estados: 8
  Ciudades: 10

🔢 Estadísticas básicas:
  Búsquedas repetidas (sum): 126,086
  Precio promedio (mean): $175.96
  Noches (median): 2

📋 Primeras 5 filas:


,city,state,country,country_code,date_start,date_end,number_of_adults,number_of_rooms,number_of_kids,nights,count_repeated,avg_hotel_count,min_hotel_count,max_hotel_count,avg_price_average,max_price_high,min_price_low,source_file
0,Seattle,Washington,United States,US,2025-12-15,2025-12-16,3,1,0,1,23,84.00,58,109,206.58,256.75,137.73,historicals_2025.csv
1,Chicago,Illinois,United States,US,2025-05-30,2025-05-31,3,2,0,1,21,170.00,118,221,129.04,171.57,90.97,historicals_2025.csv
2,New York,New York,United States,US,2025-02-28,2025-03-02,1,3,0,2,47,199.00,139,258,158.38,218.20,111.31,historicals_2025.csv
3,Chicago,Illinois,United States,US,2025-11-03,2025-11-06,2,1,2,3,14,18.00,12,23,112.89,143.64,72.95,historicals_2025.csv
4,Seattle,Washington,United States,US,2025-09-21,2025-09-22,4,1,0,1,2,143.00,100,185,132.48,160.22,94.53,historicals_2025.csv



🔍 Columnas disponibles:
['city', 'state', 'country', 'country_code', 'date_start', 'date_end', 'number_of_adults', 'number_of_rooms', 'number_of_kids', 'nights', 'count_repeated', 'avg_hotel_count', 'min_hotel_count', 'max_hotel_count', 'avg_price_average', 'max_price_high', 'min_price_low', 'source_file']


## 4️⃣ Ejecutar Pipeline Completo

In [7]:
# Ejecutar pipeline paso a paso
print("=" * 60)
print("PIPELINE: Generación de Baselines")
print("=" * 60)

# PASO 1: Cargar datos
print("\n[1/7] Cargando datos históricos...")
df_raw = auxiliary_functions.load_all_historicals()
print(f"  ✅ Cargados: {len(df_raw):,} registros")

# PASO 2: Eliminar duplicados
print("\n[2/7] Eliminando duplicados...")
initial_count = len(df_raw)
df_raw = df_raw.drop_duplicates()
print(f"  ✅ Eliminados: {initial_count - len(df_raw):,} duplicados")
print(f"  📊 Restantes: {len(df_raw):,} registros")

# PASO 3: Estandarizar precios
print("\n[3/7] Estandarizando precios...")
df_std = auxiliary_functions.standardize_prices(df_raw)
print(f"  ✅ Precio std promedio: ${df_std['avg_price_average_std'].mean():.2f}")
print(f"  📊 Rango: ${df_std['avg_price_average_std'].min():.2f} - ${df_std['avg_price_average_std'].max():.2f}")

# PASO 4: Expandir fechas
print("\n[4/7] Expandiendo fechas (multi-día → diarias)...")
df_expanded = auxiliary_functions.expand_dates_dataframe(df_std)
print(f"  ✅ Expansión: {len(df_std):,} → {len(df_expanded):,} observaciones")

# PASO 5: Features temporales
print("\n[5/7] Generando features temporales...")
df_features = auxiliary_functions.add_temporal_features(df_expanded)
print(f"  ✅ Meses únicos: {sorted(df_features['month'].unique())}")
print(f"  ✅ Semanas: {sorted(df_features['week_in_month'].unique())}")

# PASO 6: Mapear destinaciones
print("\n[6/7] Mapeando destinaciones...")
mapping_df = auxiliary_functions.load_destination_mapping()
if mapping_df is not None:
    df_mapped = auxiliary_functions.apply_destination_mapping(df_features, mapping_df)
else:
    print("  ⚠️ Sin mapping, usando ciudades directamente")
    df_mapped = df_features.copy()
    df_mapped['destination_final'] = df_mapped['city']

print(f"  ✅ Destinos finales: {df_mapped['destination_final'].nunique():,}")

# PASO 7: Calcular baselines
print("\n[7/7] Calculando baselines...")
baselines = auxiliary_functions.calculate_baselines(df_mapped)
baselines_validated = auxiliary_functions.apply_robustness_checks(baselines)

print(f"\n  ✅ Contextos totales: {len(baselines_validated):,}")
print(f"  ✅ Alta confianza: {(~baselines_validated['low_confidence']).sum():,} ({(~baselines_validated['low_confidence']).sum()/len(baselines_validated)*100:.1f}%)")
print(f"  ✅ Baja confianza: {baselines_validated['low_confidence'].sum():,} ({baselines_validated['low_confidence'].sum()/len(baselines_validated)*100:.1f}%)")

PIPELINE: Generación de Baselines

[1/7] Cargando datos históricos...

Cargando 1 archivos históricos...
  ✓ historicals_2025.csv: 5,000 registros

✓ Total cargado: 5,000 registros
  ✅ Cargados: 5,000 registros

[2/7] Eliminando duplicados...
  ✅ Eliminados: 0 duplicados
  📊 Restantes: 5,000 registros

[3/7] Estandarizando precios...

✓ Precios estandarizados: ['avg_price_average_std', 'max_price_high_std', 'min_price_low_std']
  ✅ Precio std promedio: $41.47
  📊 Rango: $7.73 - $329.00

[4/7] Expandiendo fechas (multi-día → diarias)...

Expandiendo 5,000 registros a días...
✓ Expansión: 5,000 → 18,257 observaciones diarias
  ✅ Expansión: 5,000 → 18,257 observaciones

[5/7] Generando features temporales...

✓ Features temporales generadas: month, week_in_month
  ✅ Meses únicos: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
  ✅ Semanas: [1, 2, 3, 4]

[6/7] Mapeando destinaciones...

✓ Mapping cargado: 15,988 registros
  Reducción: 13276 → 2584 destinos

✓ Mapping aplicado: 17 destinos finales


## 5️⃣ Análisis de Baselines Generados

In [8]:
# Explorar baselines generados
print("=" * 60)
print("ANÁLISIS DE BASELINES")
print("=" * 60)

print(f"\n📊 Dimensiones: {baselines_validated.shape}")
print(f"\n📋 Primeras 10 filas:")
display(baselines_validated.head(10))

print(f"\n🌍 Top 10 Destinos por Observaciones:")
top_destinations = baselines_validated.groupby('destination_final')['count_observaciones'].sum().sort_values(ascending=False).head(10)
display(top_destinations)

print(f"\n📅 Distribución por Mes:")
month_dist = baselines_validated.groupby('month').size().sort_index()
display(month_dist)

print(f"\n📆 Distribución por Semana del Mes:")
week_dist = baselines_validated.groupby('week_in_month').size().sort_index()
display(week_dist)

print(f"\n💰 Estadísticas de Precios Estandarizados:")
display(baselines_validated[['mean_price_std', 'std_price_std']].describe())

ANÁLISIS DE BASELINES

📊 Dimensiones: (816, 9)

📋 Primeras 10 filas:


,destination_final,month,week_in_month,mean_price_std,std_price_std,min_price_std,max_price_std,count_obs,low_confidence
0,77,1,1,41.47,18.14,10.88,111.24,399,False
1,77,1,2,40.28,13.39,14.81,111.24,827,False
2,77,1,3,54.88,26.18,14.81,150.68,820,False
3,77,1,4,50.72,31.61,18.61,227.00,1085,False
4,77,2,1,26.57,10.50,10.27,80.01,856,False
5,77,2,2,47.84,47.16,10.27,203.90,334,False
6,77,2,3,43.19,15.72,15.64,104.28,297,False
7,77,2,4,23.71,10.89,12.48,104.28,952,False
8,77,3,1,20.54,4.56,13.54,38.27,124,False
9,77,3,2,29.49,14.90,17.48,94.33,407,False



🌍 Top 10 Destinos por Observaciones:


KeyError: 'Column not found: count_observaciones'

## 6️⃣ Guardar Baselines

In [ ]:
# Guardar baselines a CSV
print("=" * 60)
print("GUARDANDO BASELINES")
print("=" * 60)

try:
    auxiliary_functions.save_baselines(baselines_validated)
    
    # Verificar que se guardó correctamente
    if config.BASELINES_FILE.exists():
        file_size_mb = config.BASELINES_FILE.stat().st_size / (1024 * 1024)
        print(f"\n✅ Archivo guardado: {config.BASELINES_FILE}")
        print(f"📦 Tamaño: {file_size_mb:.2f} MB")
        
        # Recargar para verificar
        baselines_reloaded = pd.read_csv(config.BASELINES_FILE)
        print(f"✅ Verificación: {len(baselines_reloaded):,} filas recargadas")
    else:
        print(f"❌ Error: Archivo no encontrado en {config.BASELINES_FILE}")
        
except Exception as e:
    print(f"❌ Error guardando: {e}")

## 7️⃣ Test de Clasificación

In [ ]:
# Test de clasificación con ejemplos
print("=" * 60)
print("TEST DE CLASIFICACIÓN")
print("=" * 60)

# Seleccionar un destino aleatorio con alta confianza
high_conf = baselines_validated[~baselines_validated['low_confidence']]
if len(high_conf) > 0:
    sample_baseline = high_conf.sample(1).iloc[0]
    
    destination = sample_baseline['destination_final']
    month = sample_baseline['month']
    week = sample_baseline['week_in_month']
    baseline_mean = sample_baseline['mean_price_std']
    baseline_std = sample_baseline['std_price_std']
    
    print(f"\n📍 Contexto de prueba:")
    print(f"  Destino: {destination}")
    print(f"  Mes: {month}")
    print(f"  Semana: {week}")
    print(f"  Baseline Mean: ${baseline_mean:.2f}")
    print(f"  Baseline Std: ${baseline_std:.2f}")
    
    # Test cases
    test_prices = [
        ("Deal", baseline_mean - 2 * baseline_std),  # 2 std por debajo
        ("Good Price", baseline_mean - 0.75 * baseline_std),  # Entre -1.0 y -0.5
        ("Normal", baseline_mean),  # En la media
        ("Expensive", baseline_mean + 0.75 * baseline_std),  # Entre 0.5 y 1.0
        ("Very Expensive", baseline_mean + 2 * baseline_std),  # 2 std por arriba
    ]
    
    print(f"\n🧪 Tests de clasificación:\n")
    
    for expected, price_std in test_prices:
        result = auxiliary_functions.evaluate_hotel_price(
            destination_final=destination,
            month=month,
            week_in_month=week,
            price_std=price_std,
            baselines_df=baselines_validated
        )
        
        classification = result['classification']
        z_score = result['z_score']
        
        match = "✅" if expected in classification else "❌"
        print(f"  {match} Precio: ${price_std:.2f} | Z: {z_score:.2f} | Clasificación: {classification}")
        
else:
    print("❌ No hay baselines de alta confianza para testear")

## 8️⃣ Resumen Final

In [ ]:
# Resumen final del sistema
print("=" * 60)
print("RESUMEN FINAL DEL SISTEMA")
print("=" * 60)

print(f"\n📊 Datos de entrada:")
print(f"  Archivos históricos cargados: {len(list(config.PRICE_HISTORICALS_DIR.glob('*.csv')))}")
print(f"  Registros raw totales: {len(df_raw):,}")
print(f"  Observaciones expandidas: {len(df_expanded):,}")

print(f"\n🎯 Baselines generados:")
print(f"  Contextos totales: {len(baselines_validated):,}")
print(f"  Destinos únicos: {baselines_validated['destination_final'].nunique():,}")
print(f"  Alta confianza: {(~baselines_validated['low_confidence']).sum():,} ({(~baselines_validated['low_confidence']).sum()/len(baselines_validated)*100:.1f}%)")
print(f"  Baja confianza: {baselines_validated['low_confidence'].sum():,} ({baselines_validated['low_confidence'].sum()/len(baselines_validated)*100:.1f}%)")

print(f"\n📁 Archivos de salida:")
if config.BASELINES_FILE.exists():
    size_mb = config.BASELINES_FILE.stat().st_size / (1024 * 1024)
    print(f"  ✅ {config.BASELINES_FILE.name} ({size_mb:.2f} MB)")
else:
    print(f"  ❌ Baselines no guardados")

print(f"\n✅ Pipeline completado exitosamente")
print(f"\n🚀 Próximo paso: Ejecutar 'streamlit run app.py'")

In [5]:
# Generar datos de muestra para testing
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

print("Generando datos de muestra...")

# Configuración
np.random.seed(42)
n_records = 5000  # Número de registros de muestra

# Ciudades de muestra
cities = ['New York', 'Los Angeles', 'Chicago', 'Miami', 'Las Vegas', 
          'San Francisco', 'Seattle', 'Boston', 'Orlando', 'Atlanta']
states = ['New York', 'California', 'Illinois', 'Florida', 'Nevada',
          'California', 'Washington', 'Massachusetts', 'Florida', 'Georgia']
countries = ['United States'] * 10
country_codes = ['US'] * 10

# Generar fechas en 2025
start_date = datetime(2025, 1, 1)
end_date = datetime(2025, 12, 31)
date_range = (end_date - start_date).days

# Generar datos
data = []
for _ in range(n_records):
    # Seleccionar ciudad
    city_idx = np.random.randint(0, len(cities))
    city = cities[city_idx]
    state = states[city_idx]
    country = countries[city_idx]
    country_code = country_codes[city_idx]
    
    # Generar fechas
    days_from_start = np.random.randint(0, date_range - 7)
    date_start_val = start_date + timedelta(days=int(days_from_start))
    nights = int(np.random.choice([1, 2, 3, 4, 5, 7], p=[0.3, 0.25, 0.2, 0.1, 0.1, 0.05]))
    date_end_val = date_start_val + timedelta(days=nights)
    
    # Parámetros de búsqueda
    number_of_adults = int(np.random.choice([1, 2, 3, 4], p=[0.2, 0.5, 0.2, 0.1]))
    number_of_rooms = int(np.random.choice([1, 2, 3], p=[0.7, 0.25, 0.05]))
    number_of_kids = int(np.random.choice([0, 1, 2], p=[0.7, 0.2, 0.1]))
    
    # Conteo repetido (demanda)
    count_repeated = int(np.random.randint(1, 50))
    
    # Métricas de hoteles
    avg_hotel_count = float(np.random.randint(10, 200))
    min_hotel_count = int(avg_hotel_count * 0.7)
    max_hotel_count = int(avg_hotel_count * 1.3)
    
    # Precios (varían por ciudad y temporada)
    base_price = {
        'New York': 180, 'Los Angeles': 160, 'Chicago': 140, 'Miami': 150,
        'Las Vegas': 120, 'San Francisco': 200, 'Seattle': 150, 
        'Boston': 170, 'Orlando': 130, 'Atlanta': 130
    }[city]
    
    # Variación estacional
    month = date_start_val.month
    seasonal_mult = 1.0
    if month in [6, 7, 8]:  # Verano
        seasonal_mult = 1.3
    elif month in [12, 1]:  # Fin de año
        seasonal_mult = 1.4
    elif month in [3, 4]:  # Primavera
        seasonal_mult = 1.1
    
    avg_price = base_price * seasonal_mult * np.random.uniform(0.8, 1.2)
    min_price = avg_price * np.random.uniform(0.6, 0.8)
    max_price = avg_price * np.random.uniform(1.2, 1.5)
    
    data.append({
        'city': city,
        'state': state,
        'country': country,
        'country_code': country_code,
        'date_start': date_start_val.strftime('%Y-%m-%d'),
        'date_end': date_end_val.strftime('%Y-%m-%d'),
        'number_of_adults': number_of_adults,
        'number_of_rooms': number_of_rooms,
        'number_of_kids': number_of_kids,
        'nights': nights,
        'count_repeated': count_repeated,
        'avg_hotel_count': avg_hotel_count,
        'min_hotel_count': min_hotel_count,
        'max_hotel_count': max_hotel_count,
        'avg_price_average': round(avg_price, 2),
        'max_price_high': round(max_price, 2),
        'min_price_low': round(min_price, 2)
    })

# Crear DataFrame
df_sample = pd.DataFrame(data)

# Guardar a CSV
output_file = config.PRICE_HISTORICALS_DIR / 'historicals_2025.csv'
df_sample.to_csv(output_file, index=False)

print(f"\n✅ Datos de muestra generados!")
print(f"  Archivo: {output_file}")
print(f"  Registros: {len(df_sample):,}")
print(f"  Ciudades: {df_sample['city'].nunique()}")
print(f"  Rango fechas: {df_sample['date_start'].min()} → {df_sample['date_start'].max()}")
print(f"\n📋 Muestra de datos:")
display(df_sample.head(10))

Generando datos de muestra...

✅ Datos de muestra generados!
  Archivo: /home/martin/Desktop/hotels/deals_analysis/data/price_historicals/historicals_2025.csv
  Registros: 5,000
  Ciudades: 10
  Rango fechas: 2025-01-01 → 2025-12-23

📋 Muestra de datos:


,city,state,country,country_code,date_start,date_end,number_of_adults,number_of_rooms,number_of_kids,nights,count_repeated,avg_hotel_count,min_hotel_count,max_hotel_count,avg_price_average,max_price_high,min_price_low
0,Seattle,Washington,United States,US,2025-12-15,2025-12-16,3,1,0,1,23,84.00,58,109,206.58,256.75,137.73
1,Chicago,Illinois,United States,US,2025-05-30,2025-05-31,3,2,0,1,21,170.00,118,221,129.04,171.57,90.97
2,New York,New York,United States,US,2025-02-28,2025-03-02,1,3,0,2,47,199.00,139,258,158.38,218.20,111.31
3,Chicago,Illinois,United States,US,2025-11-03,2025-11-06,2,1,2,3,14,18.00,12,23,112.89,143.64,72.95
4,Seattle,Washington,United States,US,2025-09-21,2025-09-22,4,1,0,1,2,143.00,100,185,132.48,160.22,94.53
5,Los Angeles,California,United States,US,2025-08-06,2025-08-08,2,2,1,2,15,199.00,139,258,193.47,247.91,131.12
6,Las Vegas,Nevada,United States,US,2025-10-07,2025-10-09,2,1,0,2,15,54.00,37,70,99.58,142.56,79.40
7,Boston,Massachusetts,United States,US,2025-03-04,2025-03-08,3,2,1,4,5,50.00,35,65,218.04,291.05,167.89
8,New York,New York,United States,US,2025-02-17,2025-02-19,2,1,0,2,22,44.00,30,57,178.00,251.69,111.06
9,New York,New York,United States,US,2025-01-05,2025-01-08,3,1,0,3,15,99.00,69,128,204.16,246.92,126.90
